In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

CATALOG = "fahrlytics"
SILVER  = f"{CATALOG}.silver"
GOLD    = f"{CATALOG}.gold"

spark.sql(f"USE CATALOG {CATALOG}")

# Temporal Cutoff Constants
CUTOFF_DATE = "2026-09-01"   # Observation window end date
HORIZON_END = "2026-10-01"   # Target window end date (30-day forward horizon)

print(f"Catalog            : {CATALOG}")
print(f"Feature Cutoff     : {CUTOFF_DATE}")
print(f"Outcome Horizon    : {CUTOFF_DATE} -> {HORIZON_END} (30 days)")
print(f"Target Gold Table  : {GOLD}.customer_features")

In [0]:
# Customer Cohort Base
print("Extracting cohort signed up on or before cutoff...")

df_cust = (
    spark.table(f"{SILVER}.customers")
    .select(
        "customer_id",
        "name",
        "city",
        "signup_date",
        "device_type",
        "referral_source",
        "rating"
    )
    .filter(F.col("signup_date") <= CUTOFF_DATE)
)

print(f"Eligible customer cohort count: {df_cust.count():,}")

In [0]:
# Dynamic Churn Target Formulation (No Leakage)
print("Computing ground-truth target label from September forward activity...")

# Identify customers who completed at least 1 trip in September 2026
september_active_customers = (
    spark.table(f"{SILVER}.trips")
    .filter(
        (F.col("trip_date") > CUTOFF_DATE) &
        (F.col("trip_date") <= HORIZON_END) &
        (F.col("trip_status") == "completed")
    )
    .select("customer_id")
    .distinct()
    .withColumn("active_in_september", F.lit(1))
)

# Label: 1 if user took NO completed trips in September, 0 if active
df_labeled_cust = (
    df_cust
    .join(september_active_customers, on="customer_id", how="left")
    .withColumn(
        "target_is_churned",
        F.when(F.col("active_in_september").isNull(), 1).otherwise(0)
    )
    .drop("active_in_september")
)

In [0]:
# Historical Trip Behavioral Features (Up to Cutoff)
print("Computing trip behavior and velocity metrics strictly prior to cutoff...")

df_trips_obs = spark.table(f"{SILVER}.trips").filter(
    (F.col("trip_date") <= CUTOFF_DATE) & (F.col("is_clean_record") == True)
)

# Preferred vehicle category window
cust_vtype_window = Window.partitionBy("customer_id").orderBy(F.col("vehicle_count").desc())

trip_fav_vehicle = (
    df_trips_obs.groupBy("customer_id", "vehicle_type")
    .agg(F.count("trip_id").alias("vehicle_count"))
    .withColumn("rank", F.row_number().over(cust_vtype_window))
    .filter(F.col("rank") == 1)
    .select("customer_id", F.col("vehicle_type").alias("preferred_vehicle_type"))
)

# Aggregate core trip signals
trip_aggregates = (
    df_trips_obs.groupBy("customer_id")
    .agg(
        # Volumes & Frequencies
        F.count("trip_id").alias("total_trips_requested"),
        F.count(F.when(F.col("trip_status") == "completed", 1)).alias("total_trips_completed"),
        F.count(F.when(F.col("trip_status") == "cancelled", 1)).alias("total_trips_cancelled"),
        
        # Financials
        F.round(F.sum(F.when(F.col("trip_status") == "completed", F.col("fare_amount"))), 2).alias("total_spend_eur"),
        F.round(F.avg(F.when(F.col("trip_status") == "completed", F.col("fare_amount"))), 2).alias("avg_fare_eur"),
        
        # Mobility & Temporal Dynamics
        F.round(F.avg(F.col("distance_km")), 2).alias("avg_distance_km"),
        F.round(F.avg(F.col("surge_multiplier")), 2).alias("avg_surge_multiplier"),
        F.round(F.avg(F.when(F.col("weather") == "Rain", 1).otherwise(0)), 2).alias("rainy_day_trip_ratio"),
        F.round(F.avg(F.when(F.col("day_type").isin("weekend", "holiday"), 1).otherwise(0)), 2).alias("weekend_trip_ratio"),
        
        # Velocity Features: August trips (last 30d of observation window)
        F.count(
            F.when(F.col("trip_date") >= F.date_sub(F.lit(CUTOFF_DATE), 30), 1)
        ).alias("trips_last_30d"),
        
        # Baseline trips: June & July (30-90 days prior to cutoff)
        F.count(
            F.when(
                (F.col("trip_date") < F.date_sub(F.lit(CUTOFF_DATE), 30)) &
                (F.col("trip_date") >= F.date_sub(F.lit(CUTOFF_DATE), 90)), 1
            )
        ).alias("trips_prior_60d")
    )
    .withColumn("cancellation_rate",
        F.round(F.col("total_trips_cancelled") / F.greatest(F.col("total_trips_requested"), F.lit(1)), 3)
    )
    # Velocity ratio: > 1.0 indicates accelerating engagement; < 1.0 indicates slowdown
    .withColumn("trip_velocity_ratio",
        F.round((F.col("trips_last_30d") * 2.0) / F.greatest(F.col("trips_prior_60d"), F.lit(1)), 2)
    )
)

In [0]:
# Payment Reliability Features (Up to Cutoff)
print("Aggregating payment reliability signals...")

df_payments_obs = spark.table(f"{SILVER}.payments").filter(
    F.to_date("payment_timestamp") <= CUTOFF_DATE
)

payment_aggregates = (
    df_payments_obs.groupBy("customer_id")
    .agg(
        F.count("payment_id").alias("total_payment_attempts"),
        F.count(F.when(F.col("payment_status") == "failed", 1)).alias("failed_payment_count"),
        F.count(F.when(F.col("is_fare_mismatch") == True, 1)).alias("fare_dispute_count"),
        F.round(F.avg(F.when(F.col("payment_method") == "PayPal", 1).otherwise(0)), 2).alias("paypal_preference_ratio")
    )
    .withColumn("payment_failure_rate",
        F.round(F.col("failed_payment_count") / F.greatest(F.col("total_payment_attempts"), F.lit(1)), 3)
    )
)

In [0]:
# Telemetry & App Funnel Features (Up to Cutoff)
print("Aggregating app clickstream telemetry...")

df_events_obs = spark.table(f"{SILVER}.clickstream").filter(
    F.col("event_date") <= CUTOFF_DATE
)

clickstream_aggregates = (
    df_events_obs.groupBy("customer_id")
    .agg(
        F.countDistinct("session_id").alias("total_app_sessions"),
        F.sum(F.when(F.col("is_search") == True, 1).otherwise(0)).alias("total_ride_searches"),
        F.sum(F.when(F.col("is_booking_confirmed") == True, 1).otherwise(0)).alias("total_confirmed_sessions"),
        F.sum(F.when(F.col("is_bounce_session") == True, 1).otherwise(0)).alias("total_bounced_sessions"),
        F.round(F.avg("session_duration_sec"), 1).alias("avg_session_duration_sec")
    )
    .withColumn("search_to_booking_conversion",
        F.round(F.col("total_confirmed_sessions") / F.greatest(F.col("total_ride_searches"), F.lit(1)), 3)
    )
    .withColumn("app_bounce_rate",
        F.round(F.col("total_bounced_sessions") / F.greatest(F.col("total_app_sessions"), F.lit(1)), 3)
    )
)

In [0]:
# Feature Matrix Assembly & Default Imputation
print("Assembling final feature vectors...")

gold_customer_features = (
    df_labeled_cust
    .join(trip_aggregates, on="customer_id", how="left")
    .join(trip_fav_vehicle, on="customer_id", how="left")
    .join(payment_aggregates, on="customer_id", how="left")
    .join(clickstream_aggregates, on="customer_id", how="left")
    .na.fill({
        "total_trips_requested": 0,
        "total_trips_completed": 0,
        "total_trips_cancelled": 0,
        "total_spend_eur": 0.0,
        "avg_fare_eur": 0.0,
        "avg_distance_km": 0.0,
        "avg_surge_multiplier": 1.0,
        "rainy_day_trip_ratio": 0.0,
        "weekend_trip_ratio": 0.0,
        "cancellation_rate": 0.0,
        "trips_last_30d": 0,
        "trips_prior_60d": 0,
        "trip_velocity_ratio": 0.0,
        "preferred_vehicle_type": "Standard",
        "total_payment_attempts": 0,
        "failed_payment_count": 0,
        "fare_dispute_count": 0,
        "payment_failure_rate": 0.0,
        "paypal_preference_ratio": 0.0,
        "total_app_sessions": 0,
        "total_ride_searches": 0,
        "total_confirmed_sessions": 0,
        "total_bounced_sessions": 0,
        "search_to_booking_conversion": 0.0,
        "app_bounce_rate": 0.0,
        "avg_session_duration_sec": 0.0
    })
    .withColumn("customer_tenure_days", F.datediff(F.lit(CUTOFF_DATE), F.col("signup_date")))
    .withColumn("_feature_calculated_at", F.current_timestamp())
)

In [0]:
# Materialize Gold Delta Table & Apply Z-ORDER
print(f"Writing feature table to {GOLD}.customer_features...")
t0 = time.time()

(
    gold_customer_features.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("city")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.customer_features")
)

print(f"Materialized Gold customer features in {round(time.time() - t0, 2)}s.")

print("Running Delta OPTIMIZE with Z-ORDER on customer_id...")
spark.sql(f"OPTIMIZE {GOLD}.customer_features ZORDER BY (customer_id)")
print("OPTIMIZE completed.")

In [0]:
# Verification & Feature Distribution Audit
print("=== FAHRLYTICS GOLD FEATURE STORE AUDIT ===\n")

df_gold_verify = spark.table(f"{GOLD}.customer_features")
total_records = df_gold_verify.count()
print(f"Total customers in Gold feature store: {total_records:,}\n")

print("Target Class Distribution (target_is_churned in September 2026):")
df_gold_verify.groupBy("target_is_churned").count().withColumn(
    "percentage", F.round(F.col("count") / total_records * 100, 2)
).show()

print("Core Feature Stats (Observation Window):")
df_gold_verify.select(
    "total_trips_completed",
    "total_spend_eur",
    "trips_last_30d",
    "trip_velocity_ratio",
    "payment_failure_rate",
    "app_bounce_rate"
).describe().show()

print("Feature Matrix Sample:")
df_gold_verify.select(
    "customer_id", "city", "customer_tenure_days",
    "trips_last_30d", "trip_velocity_ratio", "total_spend_eur",
    "target_is_churned"
).show(5, truncate=False)